# Data generation for evaluation

This notebook creates the evaluation question set for the FAQ agent and saves it to `eval/test_questions.json`.

Run this notebook first. Then run `eval/evaluations.ipynb`.

## Install dependencies once

Run this in your terminal if something is missing:

```bash
uv add requests python-frontmatter pydantic-ai pandas tqdm minsearch
```

Also make sure your `OPENAI_API_KEY` is available in the environment.

In [ ]:
from pathlib import Path
import io
import json
import random
import zipfile

import frontmatter
import requests
from pydantic import BaseModel
from pydantic_ai import Agent

# Make paths work whether the notebook is run from project root or from eval/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "eval":
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_DIR = PROJECT_ROOT / "eval"
EVAL_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Eval dir:", EVAL_DIR)

In [ ]:
def read_repo_data(repo_owner: str, repo_name: str) -> list[dict]:
    """Download and parse markdown/mdx files from a GitHub repository."""
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for file_info in zf.infolist():
            filename = file_info.filename
            filename_lower = filename.lower()

            if not (filename_lower.endswith(".md") or filename_lower.endswith(".mdx")):
                continue

            try:
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode("utf-8", errors="ignore")
                    post = frontmatter.loads(content)
                    data = post.to_dict()
                    data["filename"] = filename
                    repository_data.append(data)
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    return repository_data


dtc_faq = read_repo_data("DataTalksClub", "faq")
de_dtc_faq = [d for d in dtc_faq if "data-engineering" in d["filename"]]

print(f"Loaded {len(de_dtc_faq)} Data Engineering FAQ records")

In [ ]:
manual_questions = [
    "I just discovered the course. Can I still join?",
    "How do I install Kafka in Python?",
    "Do I need Docker for the course?",
    "Where can I find the homework?",
    "What should I do if I missed a deadline?",
    "Can I use Windows for the course?",
    "How do I submit homework?",
    "Is there a certificate?",
    "What happens if the search results do not answer my question?",
    "Can I use Python 3.12 for the course?",
]

manual_questions

In [ ]:
class QuestionsList(BaseModel):
    questions: list[str]


question_generation_prompt = """
You are helping create test questions for an AI assistant that answers questions about the DataTalksClub course FAQ.

Generate realistic user questions based on the provided FAQ records.

Include a mix of:
- easy questions
- hard questions
- beginner questions
- edge cases
- questions where the answer might not be available

Generate 15 questions.
Return only the questions.
""".strip()


question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model="gpt-4o-mini",
    output_type=QuestionsList,
)

random.seed(42)
sample_size = min(15, len(de_dtc_faq))
sample = random.sample(de_dtc_faq, sample_size)
prompt_docs = json.dumps(sample, ensure_ascii=False)

result = await question_generator.run(prompt_docs)
generated_questions = result.output.questions

for q in generated_questions:
    print(q)

In [ ]:
test_questions = manual_questions + generated_questions

output_path = EVAL_DIR / "test_questions.json"
output_path.write_text(
    json.dumps(
        {
            "manual_questions": manual_questions,
            "generated_questions": generated_questions,
            "test_questions": test_questions,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(f"Saved {len(test_questions)} evaluation questions to: {output_path}")